In [10]:
import requests
import pandas as pd
pd.set_option('display.max_columns', 50)
from io import StringIO
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
BACKEND_URL = 'http://localhost:5000'
def load_nfl_data():
    try:
        response = requests.get(f'{BACKEND_URL}/api/data/csv')
    
        if response.status_code == 200:
            # Read CSV data into pandas
            df = pd.read_csv(StringIO(response.text))
            return df
            
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None

In [12]:
nfl_data = load_nfl_data()

In [13]:
FEATURES = ['score_differential', 'ydstogo', 'yardline_100', 'play_type']
TARGET   = 'wpa'

In [14]:
df4 = nfl_data[nfl_data['down'] == 4].copy()
use_cols = FEATURES + [TARGET, 'season', 'week', 'play_id']
df4 = df4[[c for c in use_cols if c in df4.columns]].dropna(subset=[TARGET])
sort_cols = [c for c in ['season', 'week', 'play_id'] if c in df4.columns]
if sort_cols:
    df4 = df4.sort_values(sort_cols)

split_idx = int(len(df4) * 0.8)
train = df4.iloc[:split_idx].copy()
test  = df4.iloc[split_idx:].copy()

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

# --- 5) Preprocess: impute numerics, one-hot encode categoricals
num_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
cat_cols = [c for c in FEATURES if c not in num_cols]

pre = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)


In [16]:
df4['play_type'].unique()

array(['punt', 'field_goal', 'pass', 'no_play', 'run', 'qb_kneel', nan],
      dtype=object)

In [18]:
df4[df4['play_type'].isna()]['desc'].tolist()

['(11:07) (Run formation) Penalty on CAR, Delay of Game, declined.',
 '(10:50) (Shotgun) Penalty on LA, Delay of Game, declined.',
 '(:07) 16-W.Lutz 41 yard field goal is No Good, Wide Right, Center-48-M.Fraboni, Holder-9-R.Dixon. PENALTY on BUF, Defensive Too Many Men on Field, 5 yards, enforced at BUF 23.',
 '(9:22) (Punt formation) Penalty on TEN, Delay of Game, declined.',
 '(5:03) (Punt formation) Penalty on NE, Delay of Game, declined.',
 '(11:54) (Punt formation) Penalty on SEA, Delay of Game, declined.',
 '(6:11) (Punt formation) Penalty on NYJ, Delay of Game, declined.',
 '(10:30) (Punt formation) Penalty on BAL, Delay of Game, declined.']